# Main Analysis and Visualization

Core statistical analysis testing whether Afghanistan's conflict spills over to neighboring countries.

**Research Question:** Does violence in Afghanistan (t) predict violence in neighbors (t+1)?

**Methodology:**
1. Correlation analysis (Pearson r)
2. Scatter plot visualization
3. OLS lagged regression models
4. Spatial visualization

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# Define countries
NEIGHBORS = ['Turkmenistan', 'Uzbekistan', 'Tajikistan', 'Iran', 'Pakistan', 'China']

print("✓ Libraries and variables loaded")

## Load Data

Load the processed spillover dataset from the data cleaning pipeline.

In [ ]:
# Load processed data
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
data_dir = os.path.join(project_root, 'data')

try:
    data_path = os.path.join(data_dir, 'processed_spillover_data.csv')
    df = pd.read_csv(data_path)
    df['date'] = pd.to_datetime(df['date'])
    print(f"✓ Loaded analysis data: {len(df)} observations")
except Exception as e:
    print(f"Error: {e}. Please run previous notebooks first.")
    df = None

## Step 1: Correlation Analysis

**Interpretation:**
- **Negative correlation** → Concentration effect (Afghanistan's violence increases when neighbors' decline)
- **Positive correlation** → Spillover effect (Afghanistan's violence increases when neighbors' increase)
- **Values close to zero** → No spillover effect
- Heatmap color: Red (negative) to Blue (positive)

# Main analysis and visualization

This notebook runs OLS lagged regressions and saves the main visualization outputs used in the report.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

BASE_DIR = Path('/Users/chenyilu/Desktop/3148上交文件夹')
OUT_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

neighbors = ['Turkmenistan', 'Uzbekistan', 'Tajikistan', 'Iran', 'Pakistan', 'China']
df = pd.read_csv(OUT_DIR / 'afghanistan_neighbors_analysis_wide_2020.csv')
df['DATE_TS'] = pd.to_datetime(df['DATE'] + '-01')
neighbor_cols = [f'{c}_events_tplus1' for c in neighbors if f'{c}_events_tplus1' in df.columns]

## Step 1: Data Setup and Preparation

In this section, we:
- Load the cleaned dataset containing monthly aggregated political violence events
- Define the list of Afghanistan's neighboring countries
- Parse date strings and create a time-indexed column
- Prepare column references for lagged neighbor violence variables (t+1)

**Data Structure**: The dataset contains:
- One row per month (2020-02 through 2021-01, total 12 months)
- `Afghanistan_events`: violence count in Afghanistan at time t
- `{Country}_events_tplus1`: violence counts in each neighbor at time t+1

This lagged structure allows us to test whether Afghanistan's violence at month t predicts neighbors' violence at month t+1.

In [ ]:
corr_cols = ['Afghanistan_events'] + neighbor_cols
corr = df[corr_cols].corr(method='pearson')

plt.figure(figsize=(9, 7))
im = plt.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label='Pearson r')
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha='right')
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title('Correlation Heatmap: Afghanistan(t) vs Neighbors(t+1)')
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        plt.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=9)
plt.tight_layout()
heatmap_path = FIG_DIR / 'analysis_correlation_heatmap.png'
plt.savefig(heatmap_path, dpi=300)
plt.show()
print('Saved:', heatmap_path)

## Step 2: Correlation Analysis

This section calculates the **Pearson correlation coefficient** between Afghanistan's violence levels (t) and each neighbor's violence at the following month (t+1).

**What we're measuring**:
- Correlation ranges from -1 (perfect negative relationship) to +1 (perfect positive relationship)
- A **negative correlation** suggests: When Afghanistan's violence increases, neighbors' violence decreases
- A **positive correlation** suggests: When Afghanistan's violence increases, neighbors' violence also increases
- **Zero correlation** means no linear relationship

**Interpretation for spillover effects**:
- Positive correlation → Evidence of **spillover** (violence spreads to neighbors)
- Negative correlation → Evidence of **concentration** (violence concentrates in Afghanistan, away from neighbors)

The heatmap visualization makes it easy to spot patterns across all six neighboring countries simultaneously.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
axes = axes.ravel()
for idx, country in enumerate(neighbors):
    y_col = f'{country}_events_tplus1'
    ax = axes[idx]
    if y_col not in df.columns:
        ax.set_visible(False)
        continue
    valid = df[['Afghanistan_events', y_col]].dropna()
    ax.scatter(valid['Afghanistan_events'], valid[y_col], alpha=0.85)
    if len(valid) >= 2 and valid['Afghanistan_events'].var() > 0:
        slope, intercept = pd.Series(valid[y_col]).cov(valid['Afghanistan_events']) / valid['Afghanistan_events'].var(), valid[y_col].mean() - (pd.Series(valid[y_col]).cov(valid['Afghanistan_events']) / valid['Afghanistan_events'].var()) * valid['Afghanistan_events'].mean()
        x_line = pd.Series([valid['Afghanistan_events'].min(), valid['Afghanistan_events'].max()])
        y_line = intercept + slope * x_line
        ax.plot(x_line, y_line, linewidth=1.8)
    ax.set_title(country)
    ax.grid(alpha=0.25)
fig.suptitle('Scatter: Afghanistan Events (t) vs Neighbor Events (t+1)', y=1.02)
fig.supxlabel('Afghanistan events at month t')
fig.supylabel('Neighbor events at month t+1')
plt.tight_layout()
scatter_path = FIG_DIR / 'analysis_scatter_panels.png'
plt.savefig(scatter_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', scatter_path)

## Step 3: Scatter Plot Analysis

This section creates individual scatter plots for each of the six neighboring countries, with a fitted regression line through the data.

**How to read these plots**:
- **X-axis**: Afghanistan violence count at month t
- **Y-axis**: Neighbor country violence count at month t+1
- **Blue dots**: Each dot represents one month of data (12 total)
- **Red line**: Linear regression fit showing the average trend

**What the slope tells us**:
- **Steep negative slope** → Afghanistan's violence strongly predicts *decreased* neighbor violence
- **Steep positive slope** → Afghanistan's violence strongly predicts *increased* neighbor violence
- **Flat line** → No clear relationship between the variables

**Visual patterns to look for**:
- Dots clustering above/below the line indicate data spread (prediction uncertainty)
- Tight clustering around the line indicates strong predictive power
- Different countries may show very different patterns, suggesting geography/political factors matter

In [ ]:
rows = []
for country in neighbors:
    y_col = f'{country}_events_tplus1'
    if y_col not in df.columns:
        continue
    reg_df = df[['Afghanistan_events', y_col]].dropna()
    if len(reg_df) < 3:
        rows.append({'country': country, 'beta_afghanistan_events': pd.NA, 'p_value': pd.NA, 'r_squared': pd.NA, 'significant_0_05': pd.NA, 'n_obs': len(reg_df)})
        continue
    X = sm.add_constant(reg_df['Afghanistan_events'])
    y = reg_df[y_col]
    model = sm.OLS(y, X).fit()
    beta = model.params.get('Afghanistan_events', pd.NA)
    pval = model.pvalues.get('Afghanistan_events', pd.NA)
    rows.append({
        'country': country,
        'beta_afghanistan_events': beta,
        'p_value': pval,
        'r_squared': model.rsquared,
        'significant_0_05': bool(pval < 0.05) if pd.notna(pval) else pd.NA,
        'n_obs': len(reg_df),
    })

ols_summary = pd.DataFrame(rows)
ols_path = OUT_DIR / 'ols_lagged_regression_summary.csv'
ols_summary.to_csv(ols_path, index=False, encoding='utf-8-sig')
print('Saved:', ols_path)
ols_summary

## Step 4: OLS Lagged Regression Models

This is the core statistical analysis section. We run separate **Ordinary Least Squares (OLS)** regression models for each neighboring country.

**The Model**:
$$\text{NeighborViolence}_{t+1} = \beta_0 + \beta_1 \times \text{AfghanistanViolence}_t + \epsilon$$

**Key Outputs for Each Country**:

| Term | Interpretation |
|------|-----------------|
| **β (Beta)** | Slope coefficient - for every 1 additional violence event in Afghanistan at month t, how many events change in the neighbor at month t+1? |
| **p-value** | Statistical significance - values < 0.05 indicate the relationship is likely real, not due to chance |
| **R²** | Model fit - proportion of variation in neighbor violence explained by Afghanistan's violence (0 to 1 scale) |
| **Significant (0.05)** | Whether p-value is below the conventional 0.05 threshold for statistical significance |
| **N** | Sample size - number of months with valid data |

**Expected Findings**:
- **Negative β**: Afghanistan's violence predicts less violence in neighbors (concentration effect)
- **Positive β**: Afghanistan's violence predicts more violence in neighbors (spillover effect)
- **Low p-value**: We can be confident the relationship exists and isn't just random noise
- **High R²**: The model explains a large share of variation in neighbor violence

The results are saved to a CSV file for use in the final report.